# HW1 (2026): Privacy in Data Analysis

**Course:** Data Ethics 0571-4179, Tel Aviv University  
**Lecturer:** Danielle Movsowitz Davidow  
**Weight:** 15% of the final grade

**Submitted by:**

| Student name | ID |
|---|---|
|  |  |
|  |  |

<div style='padding:0.5em; background-color:#DAF7A6; color:#000000; border:1px solid #888;'>
<b>Submission instructions</b>
<ol>
<li>Due date: <b>07.06.2026</b>, 23:59.</li>
<li>This assignment counts for <b>15% of your final grade</b>.</li>
<li>Submission is in <b>pairs</b> (submit only one copy per pair).</li>
<li>Upload your solution as a <b>ZIP file</b> to the Moodle course website. The file must include:
    <ol>
    <li>A completed version of this Jupyter notebook (with all outputs visible).</li>
    <li>A <b>PDF report</b> containing all written answers (those marked <em>(report)</em>) and any plots referred to in the report.</li>
    </ol></li>
<li>Write your IDs and names in the first cell of this notebook and at the top of the PDF report.</li>
<li>Code cells are checked using automatic testing — follow function signatures and variable names exactly as specified.</li>
<li>Written answers are graded from the PDF report.</li>
<li>Your code must run without errors top-to-bottom. We will run it as part of grading.</li>
<li>You may write text in either Hebrew or English.</li>
<li>Ask questions on the Moodle Q&A forum.</li>
<li>Avoid plagiarism — cite any external source you rely on, and phrase answers in your own words.</li>
<li><b>LLM use must be documented and fully explained.</b>  If you used a language model (e.g., ChatGPT, Claude, Gemini) for any part of this assignment, include in your PDF report — for <b>every</b> use:
    <ol>
    <li>Which LLM you used (model name and version if known).</li>
    <li>What you used it for (e.g., generating code, debugging, drafting or editing report prose).</li>
    <li>The exact prompt(s) you submitted.</li>
    <li>How you verified or modified the output before relying on it.</li>
    </ol>
Undocumented or under-documented LLM use will be treated as plagiarism.</li>
</ol>
</div>

## Overview

This assignment introduces three foundational tools for privacy-preserving data analysis: **k-anonymity**, the **Laplace mechanism**, and the **exponential mechanism**.  You will see why naive anonymization fails, apply k-anonymity to a realistic publication scenario, learn how differential privacy adds calibrated noise to numeric and categorical queries, and engage with the subtleties of *sensitivity* — the property that determines how much noise is needed.

The assignment is organized in **five parts**:

1. **Part 1 — Data and privacy risk analysis.** Inspect the dataset, classify each column by its privacy role, remove direct identifiers, and articulate the privacy harms that remain.
2. **Part 2 — k-anonymity.** Demonstrate a linkage attack on the naively anonymized dataset, then apply generalization and suppression to achieve k-anonymity for a public release.
3. **Part 3 — Differential privacy: the Laplace mechanism.** Implement and apply the Laplace mechanism to numeric queries (count, sum). Investigate how the privacy budget ε controls noise.
4. **Part 4 — Sensitivity in detail.** Investigate the sensitivity of mean queries and confront the difference between **global** and **local** sensitivity. Understand why we calibrate noise to global sensitivity.
5. **Part 5 — The exponential mechanism.** Apply the exponential mechanism to a categorical query (which education level is most associated with high income). Analyze how the choice of scoring function affects both the answer and the privacy cost.

**Dataset.** The assignment uses the Adult dataset (`adult_with_pii.csv`), an adapted version of the well-known UCI dataset that includes synthetic PII columns.  The dataset is taken from Near, J.P. and Abuah, C. (2021), *Programming Differential Privacy*, https://programming-dp.com/.

---
# Part 1 — Data and Privacy Risk Analysis

Before applying any privacy-preserving technique you must first understand the data you are working with and the privacy risks it carries.  In this part you will load the dataset, classify each column by its privacy role, remove the direct identifiers, and articulate the privacy harms that remain even after that removal.

## 1.0 Imports

In [8]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# For reproducibility
np.random.seed(42)

## 1.1 Load and inspect the dataset

**Q1 (code):** Use `os.path.join(...)` to construct the path to `adult_with_pii.csv` (assume the file is in the same directory as this notebook), load it into a Pandas DataFrame named `df`, and display the first 5 rows.

In [9]:
csv_path = os.path.join(os.path.dirname('.'), 'adult_with_pii.csv')
df = pd.read_csv(csv_path)
df.head()

,Name,DOB,SSN,Zip,Age,Workclass,fnlwgt,Education,Education-Num,Marital Status,Occupation,Relationship,Race,Sex,Capital Gain,Capital Loss,Hours per week,Country,Target
0,Karrie Trusslove,9/7/1967,732-14-6110,64152,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,Brandise Tripony,6/7/1988,150-19-2766,61523,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,Brenn McNeely,8/6/1991,725-59-9860,95668,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,Dorry Poter,4/6/2009,659-57-4974,25503,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,Dick Honnan,9/16/1951,220-93-3811,75387,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


**Q2 (code):** Print:
- the shape of the dataset (number of rows and columns),
- the list of column names.

In [10]:
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Shape: (32561, 19)
Columns: ['Name', 'DOB', 'SSN', 'Zip', 'Age', 'Workclass', 'fnlwgt', 'Education', 'Education-Num', 'Marital Status', 'Occupation', 'Relationship', 'Race', 'Sex', 'Capital Gain', 'Capital Loss', 'Hours per week', 'Country', 'Target']


## 1.2 Privacy risk analysis and removal of direct identifiers

Recall from the lectures the four privacy roles a column can play: **direct identifier (PII)**, **quasi-identifier (QI)**, **sensitive attribute**, and **non-sensitive**.  In what follows you will classify each column, remove the direct identifiers, and then articulate the privacy harms that remain.

**Q3 (report):** Classify **every column** of the dataset into one of the four privacy roles, and briefly justify your choice (1 sentence per column).  Present your answer as a **table** in your PDF report with the columns: `Column name`, `Privacy role`, `Justification`.

_Write this answer in the PDF report._

**Q4 (code):** Based on your classification in Q3, create a new DataFrame `df_no_pii` from `df` by removing the columns you classified as **direct identifiers**.  Print the shape and the remaining column list of `df_no_pii` to confirm.

*Note: only direct identifiers should be removed at this stage.  Quasi-identifiers (such as zip code or age) will be addressed in Part 2.*

In [11]:
df_no_pii = df.drop(columns=['Name', 'DOB', 'SSN'])
print(f"Shape of df_no_pii: {df_no_pii.shape}")
print(f"Columns: {df_no_pii.columns.tolist()}")

Shape of df_no_pii: (32561, 16)
Columns: ['Zip', 'Age', 'Workclass', 'fnlwgt', 'Education', 'Education-Num', 'Marital Status', 'Occupation', 'Relationship', 'Race', 'Sex', 'Capital Gain', 'Capital Loss', 'Hours per week', 'Country', 'Target']


**Q5 (report):** You have just removed all direct identifiers — `df_no_pii` cannot be tied to a named individual by a single column.  Yet privacy harm is still possible.

Describe **two distinct concrete privacy harms** that could result from publishing `df_no_pii` (i.e., the dataset *after* direct identifiers have been removed, but with no further protection).  The two harms must be **different in kind** — not two examples of the same underlying mechanism.

For each harm explain:

(a) **Who is harmed** — which individuals or groups in the dataset?

(b) **What additional information** the attacker needs (if any) beyond the released dataset, and how plausible it is that they have it.

(c) **What the actual harm consists of** — be specific (e.g., re-identification, attribute disclosure, group-level inference, discrimination, etc.).

Generic statements like "privacy is violated" do not count.

_Write this answer in the PDF report._

---
# Part 2 — k-Anonymity

## Scenario

A research team has obtained this dataset (originally from the 1994 US Census) for a study on the relationship between **demographics, education, and income**.  The team plans to publicly release the dataset on a research data portal so that other academics can build on their work.  Before releasing the dataset, the team must comply with privacy regulations: individual respondents must not be re-identifiable from the released data.  Naive anonymization (Part 1) is clearly not enough.  Your job is to:

1. **Demonstrate** the residual risk of naive anonymization through a *linkage attack*.
2. **Apply k-anonymity** to prepare the dataset for safe release.
3. **Reflect** on the privacy/utility tradeoff and the limitations of k-anonymity.

Throughout this part we will treat **`Zip`, `Age`, and `Sex`** as the quasi-identifiers — the same QIs Latanya Sweeney used in her 1997 work on re-identifying Massachusetts hospital records.

## 2.1 Linkage attack on the naively anonymized data

**Q1 (code):** Implement a function `compute_k(df, quasi_identifiers)` that returns the **smallest equivalence-class size** in `df` with respect to the given list of quasi-identifier columns.  This is the *k* value of the dataset.

*Hint: group by the QI columns and look at the size of each group.*

In [12]:
def compute_k(df, quasi_identifiers):
    """
    Returns the smallest equivalence-class size in df on the given quasi-identifiers.
    """
    group_sizes = df.groupby(quasi_identifiers).size()
    return group_sizes.min()

**Q2 (code):** Compute the current k value of `df_no_pii` on the QIs `["Zip", "Age", "Sex"]` and save it in a variable `k_naive`.  Print the result.

In [13]:
k_naive = compute_k(df_no_pii, ["Zip", "Age", "Sex"])

print("k value of the naively anonymized dataset:", k_naive)

k value of the naively anonymized dataset: 1


**Q3 (code):** Compute the number of records in `df_no_pii` that are **uniquely identifiable** by their QI combination — that is, the number of rows whose `(Zip, Age, Sex)` combination appears exactly once in the dataset.  Save this number in a variable `n_unique` and print:
- the absolute number `n_unique`,
- the percentage of total records this represents (rounded to 2 decimals).

In [14]:
group_sizes = df_no_pii.groupby(["Zip", "Age", "Sex"]).size()
n_unique = (group_sizes == 1).sum()
percentage = round(100 * n_unique / len(df_no_pii), 2)

print(f"Uniquely identifiable records: {n_unique}")
print(f"Percentage of total records: {percentage}%")

Uniquely identifiable records: 32457
Percentage of total records: 99.68%


**Q4 (code + report):** In Q3 you considered an attacker who knows the full QI set `(Zip, Age, Sex)` of their target.  In practice, attackers often have only **partial** QI knowledge — for example, an acquaintance may know your age and sex but not your zip code.  In this question you will quantify how much each QI contributes to identifiability.

**(a) (code)** Compute the number of records in `df_no_pii` that would be uniquely identifiable by an attacker who knows **only `Age` and `Sex`** (i.e., does not know `Zip`).  Save the result in `n_unique_no_zip`.

**(b) (code)** Compute the number of records that would be uniquely identifiable by an attacker who knows **only `Zip` and `Sex`** (i.e., does not know `Age`).  Save the result in `n_unique_no_age`.

**(c) (report)** Compare the three numbers `n_unique`, `n_unique_no_zip`, `n_unique_no_age`.  Which of the three QIs (`Zip`, `Age`, `Sex`) carries the **most identifying power** in this dataset, and why?  Briefly relate your answer to a practical decision: *if the research team has only enough "utility budget" to generalize one QI aggressively, which one should they choose?*

_Write the discussion in (c) in the PDF report._

In [15]:
# (a) Identifiability with only {Age, Sex}
group_sizes_no_zip = df_no_pii.groupby(["Age", "Sex"]).size()
n_unique_no_zip = (group_sizes_no_zip == 1).sum()

print("Unique with {Age, Sex} only:", n_unique_no_zip)

# (b) Identifiability with only {Zip, Sex}
group_sizes_no_age = df_no_pii.groupby(["Zip", "Sex"]).size()
n_unique_no_age = (group_sizes_no_age == 1).sum()

print("Unique with {Zip, Sex} only:", n_unique_no_age)

Unique with {Age, Sex} only: 5
Unique with {Zip, Sex} only: 27086


## 2.2 Achieving k-anonymity

The team's release target is **k ≥ 5**: every QI combination in the released dataset must match at least 5 individuals.  You will achieve this in two stages — generalization, then suppression of any residual outliers.

**Q5 (code):** Generalize the `Age` column into **10-year bands**.  Concretely, create a new column `Age_band` in `df_no_pii` with values:

- ages 0–9 → `"0-9"`
- ages 10–19 → `"10-19"`
- ages 20–29 → `"20-29"`
- … and so on, up through the maximum age present in the dataset.

Then create a DataFrame `df_kanon` that is a copy of `df_no_pii` but with the original `Age` column dropped (so the QI column for age is now `Age_band`).

In [26]:
# Create Age_band: 10-year intervals
def age_to_band(age):
    """Convert age to 10-year band string"""
    band_start = (age // 10) * 10
    return f"{band_start}-{band_start+9}"

df_no_pii['Age_band'] = df_no_pii['Age'].apply(age_to_band)

# Create df_kanon: copy df_no_pii and drop Age (keep Age_band as the QI)
df_kanon = df_no_pii.drop(columns=['Age']).copy()

print(f"df_kanon shape: {df_kanon.shape}")
print(f"Age_band unique values: {sorted(df_kanon['Age_band'].unique())}")
print(f"Age_band value counts:\n{df_kanon['Age_band'].value_counts()}")

df_kanon shape: (32561, 16)
Age_band unique values: ['10-19', '20-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80-89', '90-99']
Age_band value counts:
Age_band
30-39    8613
20-29    8054
40-49    7175
50-59    4418
60-69    2015
10-19    1657
70-79     508
80-89      78
90-99      43
Name: count, dtype: int64


**Q6 (code):** Apply a second generalization: in `df_kanon`, replace the `Zip` column with its **first 3 digits** (truncate the zip code to a 3-digit prefix).  Keep the column name as `Zip`.

In [27]:
# Truncate Zip to first 3 digits
df_kanon['Zip'] = df_kanon['Zip'].astype(str).str[:3]

print(f"Zip values (sample): {df_kanon['Zip'].unique()[:10]}")

Zip values (sample): ['641' '615' '956' '255' '753' '982' '871' '608' '815' '515']


**Q7 (code):** Compute the k value of `df_kanon` on the QIs `["Zip", "Age_band", "Sex"]` and save it in `k_after_generalization`.  Print the result.

In [28]:
k_after_generalization = compute_k(df_kanon, ["Zip", "Age_band", "Sex"])

print("k after generalization:", k_after_generalization)

k after generalization: 1


**Q8 (code):** If your k from Q7 is still less than 5, generalization alone was not enough — some equivalence classes are too small.  Apply **suppression** to fix this:

1. Group `df_kanon` by `["Zip", "Age_band", "Sex"]` and compute the size of each group.
2. Identify all rows whose group size is **less than 5**.
3. Remove those rows.  Save the result as `df_kanon_5`.

Then print:
- the number of rows in `df_kanon_5`,
- the number of rows that were suppressed (relative to `df_kanon`),
- the new k value of `df_kanon_5` (it should be ≥ 5).

In [29]:
# Compute group sizes
group_sizes = df_kanon.groupby(["Zip", "Age_band", "Sex"]).size()

# Create mask for rows where group size >= 5
keep_mask = df_kanon.groupby(["Zip", "Age_band", "Sex"]).transform('size') >= 5

# Apply mask to get df_kanon_5
df_kanon_5 = df_kanon[keep_mask].copy()

# Calculate suppression statistics
rows_suppressed = len(df_kanon) - len(df_kanon_5)
k_kanon_5 = compute_k(df_kanon_5, ["Zip", "Age_band", "Sex"])

print(f"Number of rows in df_kanon_5: {len(df_kanon_5)}")
print(f"Number of rows suppressed: {rows_suppressed}")
print(f"New k value: {k_kanon_5}")

Number of rows in df_kanon_5: 17667
Number of rows suppressed: 14894
New k value: 5


**Q9 (code):** Compute the **percentage of original records lost** to suppression, rounded to 2 decimals:

$$ \text{percentage\_lost} = 100 \times \frac{|df\_no\_pii| - |df\_kanon\_5|}{|df\_no\_pii|} $$

Save the result in a variable `percentage_lost` and print it.

In [30]:
percentage_lost = round(100 * (len(df_no_pii) - len(df_kanon_5)) / len(df_no_pii), 2)

print(f"Percentage of records lost to suppression: {percentage_lost}%")

Percentage of records lost to suppression: 45.74%


## 2.3 Reflection on k-anonymity

**Q10 (code + report):** The research team now has a k-anonymous dataset (`df_kanon_5`) ready for public release.  In this question you will quantify the utility cost and reason about the choice of *k*.

**(a) (code)** Compute the **mean Capital Gain** in both `df_no_pii` and `df_kanon_5`.  Print both values and the absolute difference, rounded to 2 decimals.

**(b) (report)** A downstream researcher who only has access to `df_kanon_5` (the public release) wants to perform two analyses:

1. Mean income (`Target_num`, the fraction of `>50K` earners) by **10-year age band**.
2. Mean income by **individual age** (year-by-year).

Which of these two analyses can the downstream researcher perform on the public release, and which cannot?  In one sentence each, explain why, and explain what the difference reveals about the utility cost of k-anonymity.

**(c) (report)** The team chose `k = 5` as the target.  Suppose they had chosen `k = 20` instead.  Without re-running your code, predict whether the percentage of rows suppressed (Q9) would go **up** or **down**, and predict whether the absolute difference in mean Capital Gain (a) would be **larger** or **smaller**.  Justify each answer in one sentence.

**(d) (report)** Despite k-anonymity, an attacker who knows their target's QIs can sometimes still learn sensitive information about that target.  Describe one specific scenario in 2–3 sentences.  *Hint: an attacker may bring information beyond the QIs you generalized, or may exploit properties of the equivalence class itself.*

_Write parts (b), (c), and (d) in the PDF report._

In [31]:
# (a) Mean Capital Gain on both datasets
mean_cg_original = df_no_pii['Capital Gain'].mean()
mean_cg_kanon = df_kanon_5['Capital Gain'].mean()
difference = round(abs(mean_cg_original - mean_cg_kanon), 2)

print(f"Mean Capital Gain (df_no_pii): {round(mean_cg_original, 2)}")
print(f"Mean Capital Gain (df_kanon_5): {round(mean_cg_kanon, 2)}")
print(f"Absolute difference: {difference}")

Mean Capital Gain (df_no_pii): 1077.65
Mean Capital Gain (df_kanon_5): 1153.67
Absolute difference: 76.02


**Q11 (code + report):** In §2.2 the team chose to generalize *only* the QI set `{Zip, Age, Sex}`.  But adversaries may know other attributes about the target — `Race`, `Marital Status`, `Education`, and so on.  This question asks: how robust is the team's `k = 5` claim to that assumption?

**(a) (code)** Compute `compute_k(df_kanon_5, ...)` for three QI sets and save the results in:

- `k_three`        — QIs `['Zip', 'Age_band', 'Sex']` (the QI set the team generalized for; should be ≥ 5).
- `k_with_race`    — QIs `['Zip', 'Age_band', 'Sex', 'Race']`.
- `k_with_marital` — QIs `['Zip', 'Age_band', 'Sex', 'Race', 'Marital Status']`.

Print all three values.

**(b) (report)** Answer in the PDF report:

1. Explain in 1–2 sentences why adding more columns to the QI set can only **decrease k or leave it unchanged**, never increase it.
2. The team's privacy claim is *"every individual is hidden in a group of at least 5"*.  If an attacker happens to know the target's `Marital Status` *in addition to* `(Zip, Age, Sex)`, does this claim still hold for the released table `df_kanon_5`?  What would the team need to do to keep their claim valid against this stronger attacker?  Suggest **two distinct remedies** and briefly note the cost of each.

_Write part (b) in the PDF report._


In [32]:
# (a) Compute k with three different QI sets
k_three = compute_k(df_kanon_5, ['Zip', 'Age_band', 'Sex'])
k_with_race = compute_k(df_kanon_5, ['Zip', 'Age_band', 'Sex', 'Race'])
k_with_marital = compute_k(df_kanon_5, ['Zip', 'Age_band', 'Sex', 'Race', 'Marital Status'])

print("k with {Zip, Age_band, Sex}:", k_three)
print("k with {Zip, Age_band, Sex, Race}:", k_with_race)
print("k with {Zip, Age_band, Sex, Race, Marital Status}:", k_with_marital)

k with {Zip, Age_band, Sex}: 5
k with {Zip, Age_band, Sex, Race}: 1
k with {Zip, Age_band, Sex, Race, Marital Status}: 1


---
# Part 3 — Differential Privacy: the Laplace Mechanism

In Parts 1 and 2 you saw that even with k-anonymity, the released dataset still leaks structural information about individuals.  **Differential privacy (DP)** offers a different and stronger model: instead of transforming the dataset before release, DP transforms each *answer* to a query before returning it, by adding calibrated random noise.

Differential privacy is a formal notion of privacy applied on an *algorithm* — it is a property of algorithms, **not of the data itself**.  Simply put: an algorithm is differentially private if an observer seeing its output cannot tell whether any particular individual's information was used in the computation.  Equivalently, the algorithm produces statistically similar outputs (up to a multiplicative factor) regardless of whether any one individual's record is included.

Differential privacy is now used in many real-world products and services.  **Apple** uses local DP in its predictive keyboard.  **Google** uses DP in Chrome's metadata collection (e.g., the RAPPOR system).  **Microsoft** uses DP in Windows telemetry.  The **US Census Bureau** used DP to release the results of the 2020 Census.

## Basic terminology

1. **The privacy budget ε.**  ε is called the *privacy budget* — it acts like a knob that tunes the amount of "privacy" the definition provides.  **Lower** values of ε provide **stronger** differential-privacy protection; **higher** values provide weaker protection.  This is because there is an inverse relationship between ε and the amount of "noise" we add: smaller ε ⇒ noisier output ⇒ harder for an attacker to infer information about any single individual ⇒ more privacy.

2. **Neighboring databases.**  Two datasets D and D′ are *neighbors* if they differ by exactly one record under add/remove adjacency: |D′| = |D| ± 1, and one is obtained from the other by adding or removing exactly one row.

3. **Global sensitivity.**  The *global sensitivity* S of a numeric query f indicates that for any two neighboring datasets D and D′, the difference |f(D) − f(D′)| is at most S:

$$ S = \max_{D, D' \text{ neighbors}} |f(D) - f(D')| $$

*Worked example.*  For a count query — say, *"how many male individuals are in the dataset?"* — the global sensitivity is **1**.  For two neighboring datasets, one with individual *z* and the other without *z*, the maximum difference of the count is 1: if *z* is included in the count, removing them from the dataset decreases the count by exactly 1.

## The Laplace mechanism

The Laplace mechanism is the standard way to make a numeric query differentially private.  For a query *f* with global sensitivity *s* and privacy budget ε, the noisy answer is:

$$ F(D) = f(D) + \text{Lap}(s / \epsilon) $$

where Lap(b) is a random sample from the Laplace distribution with mean 0 and scale parameter b = s/ε.  The expected absolute size of the noise is exactly b — so larger sensitivity OR smaller ε both increase the typical noise we add.

## 3.1 The Laplace mechanism

**Q1 (code):** Implement the Laplace mechanism as a function `laplace(true_value, sensitivity, epsilon)` that returns the noisy value.  Use `np.random.laplace(loc=0, scale=...)` to sample the noise.

In [ ]:
def laplace(true_value, sensitivity, epsilon):
    """
    Returns true_value + Laplace noise with scale sensitivity / epsilon.
    """
    # -----> Write your code here <-----


## 3.2 Counting query

**Q2 (code):** Compute the **true count** of records in `df_no_pii` where `Sex == "Male"`.  Save it in `male_count` and print it.

In [ ]:
# -----> Write your code here <-----

print("True male count:", male_count)

**Q3 (code):** Apply the Laplace mechanism to `male_count` with **sensitivity = 1** and **ε = 1.0**.  Save the result in `male_count_dp` (rounded to 3 decimals) and print it.

In [ ]:
# -----> Write your code here <-----

print("Noisy male count:", male_count_dp)

## 3.3 Sum query with clipping

Counting queries are easy because their global sensitivity is always 1 — adding or removing one record can change the count by at most 1.  But for **sum** queries on numeric values like Capital Gain, the situation is different.

The Capital Gain column is an economic indicator that can span a very large range of values.  Without any bounds, an individual with an extremely high capital gain can significantly influence the sum, leading to an **unbounded** global sensitivity and therefore excessive noise when applying differential privacy.

To address this issue, we use **clipping**: we first cap each value to a chosen range [l, u] before summing.  After clipping, no single record can contribute more than (u − l) to the sum, so the global sensitivity of the clipped sum is exactly **u − l**.  We then apply the Laplace mechanism with this bounded sensitivity.

For example, if we clip Capital Gain to [0, 100000], the global sensitivity of the clipped sum is 100000.  Applying Laplace noise with ε = 1.0 gives noise of expected size 100000 — large in absolute terms, but small relative to the true sum across thousands of records.

⚠️ **Important:** the clipping bounds must be chosen *independently* of the data you are about to query — otherwise differential privacy is violated.  You will explore why in **Q6**.

**Q4 (code):** Compute the **true sum** of Capital Gain for male individuals in `df_no_pii`.  Save it in `male_cap_gain_sum` and print it.

In [ ]:
# -----> Write your code here <-----

print("True sum of Capital Gain (males):", male_cap_gain_sum)

**Q5 (code):** Apply the Laplace mechanism to a clipped version of the sum.  Procedure:

1. Clip the Capital Gain values for male individuals to the range **[0, 100000]**.
2. Compute the sum of the clipped values.
3. Apply Laplace noise with **sensitivity = 100000** and **ε = 1.0**.

Save the result in `male_cap_gain_sum_dp` and print it.

In [ ]:
# -----> Write your code here <-----

print("Noisy sum of Capital Gain (males):", male_cap_gain_sum_dp)

**Q6 (report):** In Q5 you used the clipping range [0, 100000].  Suppose instead you chose the upper bound by examining the data — for example, by setting the upper bound equal to `df_no_pii['Capital Gain'].max()`.  **What property of differential privacy would this violate, and why?**  Explain in 2–3 sentences.

_Write this answer in the PDF report._

## 3.4 The privacy budget ε

**Q7 (code):** Investigate how the privacy budget ε affects the noise magnitude of the Laplace mechanism.  For each value of ε in `[0.01, 0.1, 1.0, 10.0]`:

1. Run the noisy count of males 1000 times (sensitivity = 1, ε as given).
2. Compute the **mean absolute relative error** across the 1000 runs:  `mean( |noisy − true| / true )`.

Save the results in a dictionary `error_by_epsilon` mapping each ε to its mean relative error.  Print the dictionary.

In [ ]:
error_by_epsilon = {}
# -----> Write your code here <-----

print(error_by_epsilon)

**Q8 (report):** Looking at your `error_by_epsilon` dictionary: if you **halve ε**, what happens to the expected absolute error of the Laplace mechanism?  Justify your answer using the formula for the Laplace mechanism (one or two sentences).

_Write this answer in the PDF report._

---
# Part 4 — Sensitivity in Detail

In Part 3 you applied the Laplace mechanism to count and sum queries, where the global sensitivity was either trivial (= 1 for count) or set by the clipping bounds (= upper − lower for sum).  In this part you will look at the **mean** query, where global sensitivity is more subtle.  You will encounter the difference between **global** and **local** sensitivity, and see why DP calibrates noise to global sensitivity even when local sensitivity is much smaller.

## 4.1 The empirical impact of one row on the mean

**Q1 (code):** Build intuition by measuring the **empirical impact** of removing a single row on `mean(Age)` in `df_no_pii`.

For each row in `df_no_pii`, compute the absolute difference between `mean(Age)` of the full dataset and `mean(Age)` after removing that single row.  Save the **maximum** of these differences in `max_empirical_change` and print it (rounded to 4 decimals).

*Hint: you don't need to actually loop over rows — there is a closed-form expression.  If the full mean is `m`, the dataset has size `n`, and row i has age `a_i`, then the new mean after removing row i is `(m·n − a_i) / (n − 1)`.*

In [ ]:
# -----> Write your code here <-----

print("Max empirical change in mean(Age) by removing one row:", round(max_empirical_change, 4))

## 4.2 Global sensitivity of the mean

**Q2 (report):** Based on Q1, you might be tempted to conclude that the sensitivity of `mean(Age)` is approximately `max_empirical_change` (a small number).  But this is wrong.

Recall the formal definition of **global sensitivity**: the maximum change in the query's output across **all** neighboring database pairs — including pathological small datasets, not just `df_no_pii`.

Consider two neighboring databases, where Age is clipped to the range [0, 120]:

- D = the empty database.
- D′ = a single record with Age = 120.

What is `mean(Age)` for D and for D′ (use the convention `mean(empty) = 0`)?  What does this imply about the **global sensitivity** of `mean(Age)`?  Explain in 2–3 sentences why `max_empirical_change` (from Q1) is **not** the global sensitivity.

_Write this answer in the PDF report._

## 4.3 Local sensitivity

Local sensitivity (Near & Abuah, *Programming Differential Privacy*, Ch. 7) is defined for a *specific* dataset:

$$ LS(f, D) = \max_{D′ : d(D, D′) \le 1} |f(D) - f(D')| $$

i.e., the maximum change in *f* across neighbors of **D specifically**, not across all possible datasets.  For a mean query on a clipped value range [l, u] applied to a dataset of size n, local sensitivity has the closed form:

$$ LS_{\text{mean}}(D) = \frac{u - l}{n + 1} $$

**Q3 (code):** Compute the **local sensitivity** of `mean(Age)` on `df_no_pii`, assuming Age is clipped to [0, 120].  Save the result in `local_sens_mean_age` and print it (rounded to 6 decimals).

In [ ]:
# -----> Write your code here <-----

print("Local sensitivity of mean(Age):", round(local_sens_mean_age, 6))

**Q4 (report):** You have just observed that local sensitivity for this dataset is much smaller than global sensitivity.  So why don't we calibrate Laplace noise to local sensitivity (which would give us much more accurate answers)?

Explain in your own words (3–4 sentences) what would go wrong if a researcher published a noisy mean using **local sensitivity** instead of global sensitivity.  *Hint: think about what local sensitivity itself reveals about the dataset, and what an attacker observing the mechanism's noise scale could infer.*

_Write this answer in the PDF report._

## 4.4 Computing the mean with DP via sum/count composition

Since the global sensitivity of a mean query is too large to use directly with the Laplace mechanism, we **decompose** the mean into a sum and a count, both of which have well-defined sensitivities, and apply the Laplace mechanism separately to each.  By **sequential composition**, the total privacy budget consumed is the sum of the two individual budgets.

**Q5 (code):** Compute a differentially private estimate of `mean(Age)` for males using sum/count composition.  Use:

- ε_sum = 0.5, sensitivity_sum = 120 (Age clipped to [0, 120])
- ε_count = 0.5, sensitivity_count = 1

Procedure:
1. Compute the noisy sum of (clipped) Age for males, with the given ε_sum and sensitivity.
2. Compute the noisy count of males, with the given ε_count and sensitivity.
3. Estimate the mean as `noisy_sum / noisy_count`.

Save the result in `male_age_dp_mean` and print it (rounded to 4 decimals).

In [ ]:
# -----> Write your code here <-----

print("DP mean Age for males:", round(male_age_dp_mean, 4))

**Q6 (report):** What total ε does the procedure in Q5 satisfy?  Justify your answer by referencing the **sequential composition** property of differential privacy.

_Write this answer in the PDF report._

---
# Part 5 — The Exponential Mechanism

The Laplace mechanism (Part 3) is suitable for protecting the results of **numeric** computations such as count, sum, and mean.  But how can we protect **non-numeric** queries?  For example: *"what is the most common education level among individuals earning more than $50K?"*  The output is a category like `"Bachelors"` or `"Doctorate"`, and adding Laplace noise to a category does not make sense.

The **exponential mechanism** was proposed to handle this case — it allows selecting the *best* element from a set while preserving differential privacy.

## How the mechanism works

The analyst first defines:
- a finite set $R$ of candidate outputs to choose from (e.g., the set of all education levels), and
- a **scoring function** $u : D \times R \to \mathbb{R}$ — also called a *quality score* — that outputs a number for each candidate $r \in R$ given the dataset $D$.  Higher scores mean better candidates.

The global sensitivity of the scoring function is defined analogously to the Laplace case:

$$ \Delta u = \max_{r \in R, \; D, D' \text{ neighbors}} |u(D, r) - u(D', r)| $$

The mechanism returns candidate $r$ with probability:

$$ P[r] \propto \exp\left(\frac{\epsilon \cdot u(D, r)}{2 \Delta u}\right) $$

High-scoring candidates are exponentially more likely to be returned, with the rate of preference controlled by ε.  As with the Laplace mechanism, ε controls the privacy/utility tradeoff: smaller ε ⇒ closer to uniform random selection over R; larger ε ⇒ closer to deterministically picking the best candidate.

*Important:* the privacy cost is just ε, regardless of how large the candidate set $R$ is.  And no noise is added to the output category itself — the randomness lives in the *selection* over the candidate set.

## Worked example: most common blood type

Suppose a doctor wants to publish *"the most common blood type in their clinic"* without revealing any individual patient's blood type (sensitive medical data).  The set of candidates is:

$$ R = \{A^+, A^-, B^+, B^-, O^+, O^-, AB^+, AB^-\} $$

A natural scoring function is:

$$ u(D, r) = \text{number of patients in } D \text{ with blood type } r $$

Because this is a count, adding or removing one patient changes the score by at most 1, so $\Delta u = 1$.  Plugging this into the exponential mechanism: blood types that appear more often in the clinic data are exponentially more likely to be returned, but with some non-zero probability of returning a less common type.  The mechanism never adds noise to the categorical output itself; it samples a value from the candidate set, with probabilities tilted toward the most common ones.

## Scenario

A research team wants to publish, as part of their study on income and education, the answer to:

> **"Which education level is most strongly associated with high income (>$50K)?"**

They want to release this single category in a differentially private way, so that the published answer does not reveal anything specific about any individual respondent's data.

We compute this on `df_no_pii` (the dataset before k-anonymity was applied — the raw data the researchers analyze internally; the k-anonymous dataset of Part 2 was a separate output, intended for public release of the *full table*).

## 5.1 Implementing the exponential mechanism

**Q1 (code):** Implement the exponential mechanism as a Python function.

The function takes a dataset, a finite candidate set $R$, a scoring function, the scoring function's global sensitivity $\Delta u$, and a privacy budget $\epsilon$, and returns **one** candidate $r \in R$ — sampled with probability $P[r] \propto \exp\!\left(\dfrac{\epsilon \cdot u(D, r)}{2\,\Delta u}\right)$, as defined in the section above.

**Function signature:**

```python
def exponential(data, R, score_fn, sensitivity, epsilon):
    ...
```

**Parameters:**

| Name | Meaning |
|---|---|
| `data` | the dataset $D$ (a pandas DataFrame). |
| `R` | the candidate set — a list (or iterable) of possible outputs to choose from. |
| `score_fn` | a callable with signature `score_fn(data, r) -> float`.  For each candidate `r` it returns the score $u(D, r)$ — higher is better. |
| `sensitivity` | the global sensitivity $\Delta u$ of the scoring function. |
| `epsilon` | the privacy budget $\epsilon$. |

**Returns:** a single element of `R`, sampled according to the exponential-mechanism distribution.

**Implementation steps:**

1. For each candidate `r ∈ R`, compute its score `score_fn(data, r)`.  Collect the scores into a numpy array.
2. Compute the exponent `e_r = ε · score(r) / (2 · sensitivity)` for each candidate.  We will call these the *log-weights*, because $\exp(e_r)$ is the un-normalized weight that determines $P[r]$.
3. **Numerical-stability step.**  Subtract `max(e_r)` from every log-weight before exponentiating.  This shifts every exponent by the same constant, so after normalization the probabilities are unchanged — but it keeps the largest exponent at $0$ and prevents `np.exp(...)` from overflowing on large positive arguments.  (This is the standard *log-sum-exp* trick.)
4. Apply `np.exp(...)` to the shifted log-weights, then normalize so they sum to $1$ — these are the sampling probabilities.
5. Return one element from `R` using `np.random.choice(R, p=probabilities)`.


In [ ]:
def exponential(data, R, score_fn, sensitivity, epsilon):
    """
    Returns one element from R, sampled via the exponential mechanism.
    """
    # -----> Write your code here <-----


## 5.2 The scoring function

The natural scoring function for *"which education level is most associated with high income"* is:

$$ u(D, e) = \text{count of records in } D \text{ where Education = } e \text{ AND Target = } \texttt{>50K} $$

This scores each education level by the number of high earners with that education.  The candidate with the highest score is the *most common education level among high earners*.

**Q2 (code):** Implement the scoring function `score_education(data, education_level)` that returns the count of rows in `data` where `Education == education_level` AND `Target == ">50K"`.

In [ ]:
def score_education(data, education_level):
    # -----> Write your code here <-----


**Q3 (code):** What is the **global sensitivity** Δu of `score_education`?

Recall: Δu = max |u(D, r) − u(D′, r)| over all neighboring pairs (D, D′), for any fixed r.  The neighbor differs by exactly one record.  How much can adding/removing one record change a count?

Save your answer (a single number) in the variable `delta_u` and print it.

In [ ]:
# -----> Write your answer here <-----

print("Δu of score_education:", delta_u)

**Q4 (code):** Now apply the mechanism:

1. Compute the unique education levels in `df_no_pii` and save them in a list `education_options`.
2. Apply the exponential mechanism with **ε = 1.0** to find the differentially private most-common high-earner education.  Save the result in `dp_top_education`.
3. Compute the **true** most-common high-earner education (without DP).  Save it in `true_top_education`.
4. Print both.

In [ ]:
# -----> Write your code here <-----

print("True top education among high earners:", true_top_education)
print("DP top education (one run):", dp_top_education)

## 5.3 Choosing the scoring function — analysis

The scoring function is a **design choice**.  We chose `u(D, e) = count of >50K with Education = e`.  An alternative would be:

$$ u_{\text{alt}}(D, e) = \frac{\text{count of >50K with Education = }e}{\text{count of records with Education = }e} $$

That is, u_alt is the **rate** of high earners per education level, rather than the count.

**Q5 (report):** Compare the two scoring functions u (count) and u_alt (rate):

**(a)** Conceptually, which scoring function more accurately captures the question *"which education level is most associated with high income"*?  Justify in 1–2 sentences.

**(b)** What is the **global sensitivity Δu_alt** of u_alt?  *Hint: consider the worst case where a single record is added to a dataset that has very few records of that education level.*  Connect your answer to what you observed about **mean-query sensitivity in Part 4** — what do mean queries and rate-based scoring functions have in common?

**(c)** Even though u_alt is conceptually better aligned with the question, why is using u (count) the **practical** choice for the exponential mechanism in this dataset?  *Hint: think about the score range and how it interacts with the noise scale ε / (2 · Δu).*

_Write this answer in the PDF report._

## 5.4 Privacy budget ε vs. accuracy

**Q6 (code):** Investigate how the privacy budget ε affects the accuracy of the exponential mechanism's output:

1. Generate 20 values of ε using `np.logspace(-3, 1, 20)`.
2. For each ε, run the exponential mechanism (with `score_education`, `delta_u`, and the given ε) **200 times**.  Record the **accuracy** as the fraction of runs whose output equals `true_top_education`.
3. Save the list of accuracies (in the same order as `epsilons`) in the variable `accuracies`.

*This may take a minute or two to run.*

In [ ]:
epsilons = np.logspace(-3, 1, 20)
accuracies = []
# -----> Write your code here <-----


**Q7 (code):** Plot accuracy (y-axis) vs ε (x-axis) using a **log-scaled x-axis** (`plt.xscale('log')`).  Label both axes and add a title.

_Include this plot in your PDF report._

In [ ]:
# -----> Write your code here <-----


**Q8 (report):** Looking at your accuracy-vs-ε plot:

**(a)** Roughly at what ε does the mechanism reach > 90% accuracy?  At what ε is its output close to a uniformly random guess?

**(b)** The research team is preparing to **publish** the result of this DP query.  Recommend a value of ε they should use, and **justify** your recommendation: it should be small enough to provide meaningful privacy, but large enough to give a reliable answer.  There is no single right answer — make a reasoned argument referring to your accuracy curve.

_Write this answer in the PDF report._

---
## Final submission checklist

- [ ] All cells in this notebook run from top to bottom without errors.
- [ ] The notebook contains your IDs and names in the first cell.
- [ ] All `(code)` questions have working code.
- [ ] All `(report)` questions are answered in your PDF report.
- [ ] The PDF report begins with your IDs and names.
- [ ] The submission is a single ZIP file containing the completed notebook and the PDF.

Good luck!